# Tesla and GameStop Stock & Revenue Analysis

**IBM Finance and Web Scraping Final Project**

Run the cells from top to bottom in Google Colab or IBM Skills Network Jupyter. This notebook contains the code for Questions 1–6. After running each question, take screenshots of the relevant code cell(s) together with the actual output.

> Important: outputs are generated when you run the notebook. Do not submit screenshots of unexecuted code or sample/fabricated results.


## Setup — install required libraries

This notebook is designed for Google Colab, which runs in your browser and does not require installing Python on your computer. Run this cell first.

In [ ]:
%pip -q install yfinance pandas requests beautifulsoup4 html5lib plotly

## Imports

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)


## Question 1 — Extract Tesla stock data using yfinance

Create the Tesla ticker, download the maximum available historical data, reset the index, and display the first five rows.

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2 — Extract Tesla revenue data using web scraping

Fetch the course-provided Tesla revenue webpage, parse its HTML, locate the revenue table, clean the values, and display the last five rows.

In [ ]:
tesla_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
response = requests.get(tesla_url, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html5lib")

def extract_revenue_table(soup):
    # Find the table whose header contains both Date and Revenue.
    for table in soup.find_all("table"):
        headers = [th.get_text(" ", strip=True).lower()
                   for th in table.find_all(["th"])]
        if "date" in headers and "revenue" in headers:
            rows = []
            for tr in table.find_all("tr"):
                cols = tr.find_all("td")
                if len(cols) >= 2:
                    rows.append([
                        cols[0].get_text(strip=True),
                        cols[1].get_text(strip=True)
                    ])
            if rows:
                return pd.DataFrame(rows, columns=["Date", "Revenue"])

    # Fallback for course pages whose table has no explicit header cells.
    for table in soup.find_all("table"):
        rows = []
        for tr in table.find_all("tr"):
            cols = tr.find_all("td")
            if len(cols) >= 2:
                rows.append([
                    cols[0].get_text(strip=True),
                    cols[1].get_text(strip=True)
                ])
        if rows:
            return pd.DataFrame(rows, columns=["Date", "Revenue"])
    raise ValueError("Could not find a two-column revenue table.")

tesla_revenue = extract_revenue_table(soup)
tesla_revenue["Revenue"] = (
    tesla_revenue["Revenue"]
    .str.replace(r"[$,]", "", regex=True)
    .str.strip()
)
tesla_revenue = tesla_revenue[tesla_revenue["Revenue"] != ""]
tesla_revenue["Revenue"] = pd.to_numeric(tesla_revenue["Revenue"], errors="coerce")
tesla_revenue = tesla_revenue.dropna(subset=["Revenue"])
tesla_revenue.tail()

## Question 3 — Extract GameStop stock data using yfinance

Create the GameStop ticker, download the maximum available historical data, reset the index, and display the first five rows.

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4 — Extract GameStop revenue data using web scraping

Fetch the course-provided GameStop revenue webpage, parse its HTML, clean the revenue values, and display the last five rows.

In [ ]:
gme_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
response = requests.get(gme_url, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html5lib")
gme_revenue = extract_revenue_table(soup)
gme_revenue["Revenue"] = (
    gme_revenue["Revenue"]
    .str.replace(r"[$,]", "", regex=True)
    .str.strip()
)
gme_revenue = gme_revenue[gme_revenue["Revenue"] != ""]
gme_revenue["Revenue"] = pd.to_numeric(gme_revenue["Revenue"], errors="coerce")
gme_revenue = gme_revenue.dropna(subset=["Revenue"])
gme_revenue.tail()

## Dashboard function

This function is used for Questions 5 and 6. It plots the share price and revenue in two panels.

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    stock_plot = stock_data.copy()
    revenue_plot = revenue_data.copy()

    stock_plot["Date"] = pd.to_datetime(stock_plot["Date"])
    revenue_plot["Date"] = pd.to_datetime(revenue_plot["Date"])

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Historical Revenue"),
        vertical_spacing=0.25
    )

    fig.add_trace(
        go.Scatter(x=stock_plot["Date"], y=stock_plot["Close"],
                   name="Share Price"),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=revenue_plot["Date"], y=revenue_plot["Revenue"],
                   name="Revenue"),
        row=2, col=1
    )

    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Share Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD)", row=2, col=1)
    fig.update_layout(
        title=f"{stock} Stock and Revenue",
        height=850,
        showlegend=True
    )
    fig.show()


## Question 5 — Tesla stock and revenue dashboard

Run this cell and take a screenshot showing the complete chart.

In [ ]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6 — GameStop stock and revenue dashboard

Run this cell and take a screenshot showing the complete chart.

In [ ]:
make_graph(gme_data, gme_revenue, "GameStop")

## Question 7 — Save and upload this notebook to GitHub

1. In Google Colab, choose **File → Download → Download .ipynb** after running and saving the notebook.
2. Sign in to GitHub and create a public repository named `Tesla-GameStop-Stock-Analysis`.
3. In the repository, choose **Add file → Upload files**.
4. Upload this `.ipynb` notebook, then click **Commit changes**.
5. Copy the repository URL and paste it into the assignment's GitHub URL field.

Submit your own executed notebook and screenshots of the actual outputs. This starter notebook itself is not evidence that the code ran successfully.
